In [1]:
%load_ext autoreload
%autoreload 2
    
import os
import sys
import pandas as pd
import xarray as xr
import numpy as np
from dask.distributed import Client
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

from cdrmip_extremes import processing, utils, stats
from cdrmip_extremes.configs import DATA_DIR, RAW_DIR, PROC_DIR, CDR_MODELS as models, CDR_EXPTS as expts

In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/37375/status,
Dashboard: /proxy/37375/status,Workers: 28
Total threads: 28,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:32919,Workers: 0
Dashboard: /proxy/37375/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:42685,Total threads: 1
Dashboard: /proxy/35943/status,Memory: 0 B
Nanny: tcp://127.0.0.1:43789,


In [3]:
msftmz_dir = RAW_DIR / 'msftmz-msftyz'
msftmz = {model:{} for model in models}
for model in models:
    for expt in expts:
        path = os.path.join(msftmz_dir,expt,model,'*.nc')
        msftmz[model][expt] = xr.open_mfdataset(
            path,
            use_cftime=True,
            drop_variables = ['lev_bnds'],
            chunks={'time':120}
        )

In [5]:
concat = {}
for model in models:
    ds_up = msftmz[model]['1pctCO2']
    ds_down = msftmz[model]['1pctCO2-cdr']
    concat[model] = processing.concatenate_branches(
        child_ds=ds_down,
        parent_ds=ds_up,
        realign_years=(0,140)
    )

Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0140-01-15 12:00:00, child relabeled to start 0140-01-15 12:45:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 013

## Check model coordinates ##

In [17]:
amoc_configs = {model:{} for model in models}
for model in models:
    print(model)
    coords = list(concat[model].coords)
    lat_coord = [coord for coord in coords if coord in ['lat','rlat','y','j-mean']][0]
    variables = list(concat[model].variables)
    print(lat_coord)
    amoc_configs[model]['coord'] = lat_coord
    amoc_configs[model]['var'] = [variable for variable in variables if variable in ['msftmz','msftyz']][0]

ACCESS-ESM1-5
lat
CanESM5
lat
CESM2
lat
CNRM-ESM2-1
j-mean
GFDL-ESM4
y
MIROC-ES2L
lat
NorESM2-LM
lat
UKESM1-0-LL
rlat


## Calculate AMOC

In [18]:
def calc_amoc(str_func, basin_no, coord, lat):
    
    # isolate latitude of 26.5N in atlantic ocean
    str_func_26N = str_func.sel(
        basin=basin_no,
        drop=True
    ).sel(
        **{coord:lat},
        method='nearest',
        drop=True
    )
    
    # identify depth with maximum stream function
    # and select only these depths
    
    max_lev_idx = str_func_26N.idxmax(dim='lev')
    amoc_26N = str_func_26N.sel(lev=max_lev_idx,drop=True) / 1.025e9
    
    return amoc_26N

In [19]:
amoc_26N_dict = {}
for model in models:
    str_func = concat[model]
    # CNRM-ESM1-1 has different basin and latitude coordinates
    if model == 'CNRM-ESM2-1':
        basin_no = 1
        lat = 190.283
    else:
        basin_no = 0
        lat=26.5
    coord = amoc_configs[model]['coord']
    variable = amoc_configs[model]['var']
    print(coord)
    amoc_26N_dict[model] = calc_amoc(str_func[variable],basin_no,coord,lat)

lat
lat
lat
j-mean
y
lat
lat
rlat


In [10]:
# save #
save_dir = PROC_DIR / 'CDRMIP' / 'amoc'
os.makedirs(save_dir,exist_ok=True)

for model, ds in amoc_26N_dict.items():
    path = save_dir / f"{model}_amoc_26N.nc"
    print(f"Saving to {path}")
    ds.to_netcdf(path)

Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/ACCESS-ESM1-5_amoc_26N.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CanESM5_amoc_26N.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CESM2_amoc_26N.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CNRM-ESM2-1_amoc_26N.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/GFDL-ESM4_amoc_26N.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/MIROC-ES2L_amoc_26N.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/NorESM2-LM_amoc_26N.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/UKESM1-0-LL_amoc_26N.nc


### and for piControl ###

In [11]:
amoc_pi_dict = {}
for model in models:

    str_func = msftmz[model]['piControl']
    # CNRM-ESM1-1 has different basin and latitude coordinates
    if model == 'CNRM-ESM2-1':
        basin_no = 1
        lat = 190.283
    else:
        basin_no = 0
        lat=26.5
    coord = amoc_configs[model]['coord']
    variable = amoc_configs[model]['var']
    print(coord)
    amoc_pi_dict[model] = calc_amoc(str_func[variable],basin_no,coord,lat)

lat
lat
lat
j-mean
y
lat
lat
rlat


In [13]:
# save #
save_dir = PROC_DIR / 'CDRMIP' / 'amoc'

for model, ds in amoc_pi_dict.items():
    path = save_dir / f"{model}_amoc_26N_piControl.nc"
    print(f"Saving to {path}")
    ds.to_netcdf(path)

Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/ACCESS-ESM1-5_amoc_26N_piControl.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CanESM5_amoc_26N_piControl.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CESM2_amoc_26N_piControl.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CNRM-ESM2-1_amoc_26N_piControl.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/GFDL-ESM4_amoc_26N_piControl.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/MIROC-ES2L_amoc_26N_piControl.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/NorESM2-LM_amoc_26N_piControl.nc
Saving to /home/563/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/UKESM1-0-LL_amoc_26N_piControl.nc
